## Overall syllabus
* https://docs.pytorch.org/tutorials/beginner/basics/intro.html
* https://docs.pytorch.org/tutorials/distributed.html
* https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html
* Individual Topics
  - Implement linear regression using neural network with gradients and stuff
  - Implement lora from scratch
  - Knowledge Distillation (https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html)
  - Activation Recomp
  - Attention (MHA, Grouped, etc)
  - Flash Attention
  - Fine-tuning, chopping off stuff
  - GradAcc
  - AMP during forwar pass, scaling the gradients, optimizer etc

Nice to haves
* https://docs.pytorch.org/tutorials/compilers_index.html

Intrview Prep Resources
- https://www.deep-ml.com/deep-0
- https://github.com/Exorust/TorchLeet
- https://github.com/Devinterview-io/pytorch-interview-questions
- https://hackmd.io/@husseinsheikho/pytorch-interview?utm_source=chatgpt.com

In [ ]:
# https://docs.pytorch.org/tutorials/beginner/basics/intro.html


## Learnings
- PyTorch tracks gradients using `.grad` only for leaf tensors like weights, biases, inputs, etc for memory reasons. Non leaf tensor gradients are automatically not tracked. If a non leaf tensor has a tensor with `requires_grad=True` in its computation, then by default it's `requires_grad` will also be `True`. However, at the end of the backward pass `.grad` of the non leaf tensor will be released from memory because it is not needed.
- Only non-leaf tensors have the `grad_fn` to flow to the previous guy, leaf tenosrs do not have `grad_fn`
- If you want to track gradients for non leaf tensors, you can do `z.retain_grad`() after calculating `z = w @ x + b`. Then the `.grad` will be saved.
- We can only perform gradient calculations using `backward` once on a given graph, for performance reasons. If we need to do several `backward` calls on the same graph, we need to pass `retain_graph=True` to the backward call.
- PyTorch accumulates gradients for leaf tensors. If you want to call `backward` again, then you should zero them out using `inp.grad = torch.zeros_like(inp.grad)` or simply `inp.grad.zero_()`
- When we call forward, we esentially do 2 things, 1) compute the output 2) compute the grad_fn of the tensor and place it in the DAG which will be used during the backward pass
- Different types of grad modes:
  - default: normal stuff, used in forward pass, tensors created in this mode can be used in grad-mode later
  - no-grad: excludes operations from being recorded in backward pass, tensors created in this mode can be used in grad-mode later (like before)
  - inference: excludes operations from being recorded in backward pass, tensors created in this mode CANNOT be used in grad-mode later
  - model.eval(): this mode is used when you have dropout and batchnorm layers so that for validation data you do not calculate stats and for dropout you dont dropout. use model.train() for training models
  - NOTE: YOU NEED TO SET MODEL.EVAL() AND TORCH.NO_GRAD both for CNN models, they do diff things, details in optimizer section below

In [ ]:
# basics

In [ ]:
import torch

In [ ]:
shape = (4,5)

In [ ]:
tensor = torch.rand(shape, device='cuda')

In [ ]:
tensor

tensor([[0.7405, 0.8874, 0.9420, 0.3098, 0.2166],
        [0.7273, 0.8799, 0.8365, 0.2206, 0.5738],
        [0.7338, 0.1009, 0.9080, 0.7615, 0.5039],
        [0.8206, 0.3753, 0.0232, 0.2182, 0.7147]], device='cuda:0')

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
t2 = torch.cat([tensor, tensor, tensor], dim=0)

In [ ]:
t1.shape, t2.shape

(torch.Size([4, 15]), torch.Size([12, 5]))

In [ ]:
y1 = tensor @ tensor.T # matmul
y2 = tensor * tensor # element-wise sum

In [ ]:
y1.shape, y2.shape

(torch.Size([4, 4]), torch.Size([4, 5]))

In [ ]:
agg = tensor.sum()

In [ ]:
agg.item()

11.494308471679688

In [ ]:
y3 = tensor + 5

In [ ]:
y3

tensor([[5.7405, 5.8874, 5.9420, 5.3098, 5.2166],
        [5.7273, 5.8799, 5.8365, 5.2206, 5.5738],
        [5.7338, 5.1009, 5.9080, 5.7615, 5.5039],
        [5.8206, 5.3753, 5.0232, 5.2182, 5.7147]], device='cuda:0')

In [ ]:
tensor_copy = torch.zeros_like(tensor)

In [ ]:
tensor_copy.add_(5)

tensor([[5., 5., 5., 5., 5.],
        [5., 5., 5., 5., 5.],
        [5., 5., 5., 5., 5.],
        [5., 5., 5., 5., 5.]], device='cuda:0')

In [ ]:
# datasets and dataloaders

In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.77MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 16.0MB/s]


In [ ]:
train_dataloader = DataLoader(training_data, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=4, shuffle=True)

In [ ]:
for x, y in train_dataloader:
  print(x.shape, y.shape)
  break

torch.Size([4, 1, 28, 28]) torch.Size([4])


In [ ]:
# simple training

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [ ]:
device

'cuda'

In [ ]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    x = self.linear_relu_stack(x)
    return x

model = NeuralNetwork().to(device)

In [ ]:
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
shape = (1, 28, 28)
X = torch.rand(shape)
X.shape

torch.Size([1, 28, 28])

In [ ]:
X = X.to(device)
logits = model(X)

In [ ]:
logits

tensor([[-0.1115, -0.0631,  0.0014,  0.0334, -0.0709, -0.1075,  0.0785,  0.0763,
         -0.0386,  0.0304]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [ ]:
pred_probs = torch.nn.functional.softmax(logits, dim=1)
pred_probs

tensor([[0.0908, 0.0953, 0.1016, 0.1049, 0.0946, 0.0912, 0.1098, 0.1095, 0.0977,
         0.1046]], device='cuda:0', grad_fn=<SoftmaxBackward0>)

In [ ]:
y_pred = pred_probs.argmax()

In [ ]:
y_pred

tensor(6, device='cuda:0')

In [ ]:
x = torch.rand(3,28,28)
flattened_image = nn.Flatten()(x)

In [ ]:
flattened_image.shape

torch.Size([3, 784])

In [ ]:
linear_layer = nn.Linear(28*28, 50)
lin_output = linear_layer(flattened_image)

In [ ]:
lin_output.shape

torch.Size([3, 50])

In [ ]:
relu_op = nn.ReLU()(lin_output)

In [ ]:
relu_op.size()

torch.Size([3, 50])

In [ ]:
lin_output.shape

torch.Size([3, 50])

In [ ]:
for name, param in model.named_parameters():
  print(name, param.shape)

linear_relu_stack.0.weight torch.Size([512, 784])
linear_relu_stack.0.bias torch.Size([512])
linear_relu_stack.2.weight torch.Size([512, 512])
linear_relu_stack.2.bias torch.Size([512])
linear_relu_stack.4.weight torch.Size([10, 512])
linear_relu_stack.4.bias torch.Size([10])


In [ ]:
model

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)

In [ ]:
# autograd

In [ ]:
import torch

In [ ]:
x = torch.ones(5) # input tensor
y = torch.zeros(3) # expected output

In [ ]:
w = torch.rand((5,3), requires_grad=True)
b = torch.rand(3, requires_grad=True)

In [ ]:
z = x @ w + b
# z.retain_grad() # this is the way to retain gradients of non-leaf tensors,
# pytorch doesnt retain gradients of non-leaf tensors for memory efficiency

In [ ]:
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
loss

tensor(3.5078, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

In [ ]:
# tomorrow: forget about binary/multi-class CE loss and learn the grad
# stuff starting from this section: https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html#tensors-functions-and-computational-graph

In [ ]:
loss.grad_fn

In [ ]:
z.grad_fn

In [ ]:
w.grad, b.grad

(None, None)

In [ ]:
loss.backward()

In [ ]:
w.grad, b.grad

(tensor([[0.3238, 0.3254, 0.3202],
         [0.3238, 0.3254, 0.3202],
         [0.3238, 0.3254, 0.3202],
         [0.3238, 0.3254, 0.3202],
         [0.3238, 0.3254, 0.3202]]),
 tensor([0.3238, 0.3254, 0.3202]))

In [ ]:
z.grad # gradients only present for leaf tensors

/tmp/ipython-input-2280635533.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  z.grad # gradients only present for leaf tensors


In [ ]:
z.requires_grad

True

In [ ]:
with torch.no_grad():
  z = x @ w + b

In [ ]:
z.requires_grad

False

In [ ]:
# advanced: tensor gradients and jacobians

In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
inp

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.]], requires_grad=True)

In [ ]:
out = (inp+1).pow(2).t()
out

tensor([[4., 1., 1., 1.],
        [1., 4., 1., 1.],
        [1., 1., 4., 1.],
        [1., 1., 1., 4.],
        [1., 1., 1., 1.]], grad_fn=<TBackward0>)

In [ ]:
print(inp.grad) # will be None
print(out.grad) # will be None + will show warning that PyTorch doesnt track gradients for non leaf tensors

None
None


/tmp/ipython-input-3271208308.py:2: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(out.grad) # will show warning that PyTorch doesnt track gradients for non leaf tensors


In [ ]:
torch.ones_like(out)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [ ]:
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [ ]:
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")


Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])


In [ ]:
inp.grad.zero_()

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [ ]:
torch.zeros_like(inp)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [ ]:
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")


Second call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [ ]:
# auto grad mechanics

In [ ]:
# division by zero
x = torch.tensor([1. ,1.], requires_grad=True)
div = torch.tensor([0., 1.])

In [ ]:
y = x / div

In [ ]:
y

tensor([inf, 1.], grad_fn=<DivBackward0>)

In [ ]:
mask = div != 0

In [ ]:
mask

tensor([False,  True])

In [ ]:
loss = y[mask].sum()

In [ ]:
loss.grad_fn

In [ ]:
loss.backward()

In [ ]:
x.grad # leads to unstable training

tensor([nan, 1.])

In [ ]:
# safe way
x = torch.tensor([1., 1.], requires_grad=True)
div = torch.tensor([0., 1.])

mask = div != 0
safe = torch.zeros_like(x)
safe[mask] = x[mask] / div[mask]
loss = safe.sum()

In [ ]:
loss.backward()

In [ ]:
x.grad

tensor([0., 1.])

In [ ]:
# Optimizing Model Parameters

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 194kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.59MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.0MB/s]


In [3]:
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [4]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

In [5]:
model = NeuralNetwork()
model = model.to('cuda')

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
  dataset_size = len(dataloader.dataset)

  model.train()
  for batch_id, (X, y) in enumerate(dataloader):
    X = X.to('cuda')
    y = y.to('cuda')

    logits = model(X)
    loss = loss_fn(logits, y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch_id % 100 == 0:
      loss, current = loss.item(), batch_id * batch_size + len(X)
      print(f"loss: {loss:>7f}  [{current:>5d}/{dataset_size:>5d}]")

def test_loop(dataloader, model, loss_fn):
  model.eval()
  dataset_size = len(dataloader.dataset)
  num_batches = len(dataloader)

  loss, correct = 0, 0
  with torch.no_grad():
    for X, y in dataloader:
      X = X.to('cuda')
      y = y.to('cuda')
      pred = model(X)
      loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  loss /= num_batches
  correct /= dataset_size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {loss:>8f} \n")

In [ ]:
epochs = 2
for i in range(epochs):
  print(f"Epoch {i+1}\n------")
  train_loop(train_dataloader, model, loss_fn, optimizer)
  test_loop(test_dataloader, model, loss_fn)

Epoch 1
------
loss: 2.306509  [   64/60000]
loss: 0.548228  [ 6464/60000]
loss: 0.394674  [12864/60000]
loss: 0.505407  [19264/60000]
loss: 0.453435  [25664/60000]
loss: 0.427422  [32064/60000]
loss: 0.372685  [38464/60000]
loss: 0.535359  [44864/60000]
loss: 0.499144  [51264/60000]
loss: 0.523920  [57664/60000]
Test Error: 
 Accuracy: 84.6%, Avg loss: 0.427234 

Epoch 2
------
loss: 0.273438  [   64/60000]
loss: 0.358075  [ 6464/60000]
loss: 0.278174  [12864/60000]
loss: 0.383861  [19264/60000]
loss: 0.405907  [25664/60000]
loss: 0.386022  [32064/60000]
loss: 0.325180  [38464/60000]
loss: 0.479053  [44864/60000]
loss: 0.378863  [51264/60000]
loss: 0.462539  [57664/60000]
Test Error: 
 Accuracy: 85.6%, Avg loss: 0.395087 



In [ ]:
model.state_dict().keys()

odict_keys(['linear_relu_stack.0.weight', 'linear_relu_stack.0.bias', 'linear_relu_stack.2.weight', 'linear_relu_stack.2.bias', 'linear_relu_stack.4.weight', 'linear_relu_stack.4.bias'])

In [ ]:
model

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)

In [ ]:
optimizer.state_dict()

{'state': {0: {'step': tensor(1876.),
   'exp_avg': tensor([[-5.6052e-45, -7.9087e-18, -7.9087e-18,  ..., -1.7575e-05,
            -1.5960e-06, -1.1142e-08],
           [ 5.6052e-45, -4.3206e-11, -4.3206e-11,  ..., -1.9458e-05,
            -1.9863e-06, -1.3464e-08],
           [-5.6052e-45, -5.6052e-45,  6.6513e-37,  ..., -3.8257e-06,
            -4.7210e-42, -5.6052e-45],
           ...,
           [ 0.0000e+00,  5.6052e-45,  5.6052e-45,  ..., -2.8521e-11,
            -3.6365e-10, -4.5682e-43],
           [ 5.6052e-45,  1.0288e-07,  3.0089e-07,  ..., -3.9293e-07,
             1.1770e-07,  1.9176e-08],
           [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  5.6052e-45,
             0.0000e+00,  0.0000e+00]], device='cuda:0'),
   'exp_avg_sq': tensor([[2.8818e-14, 6.6150e-13, 6.8260e-13,  ..., 1.1444e-08, 2.3309e-09,
            1.6501e-10],
           [4.4530e-17, 7.1023e-13, 8.6959e-13,  ..., 3.7205e-08, 1.1921e-08,
            3.7147e-10],
           [1.5391e-13, 1.5463e-13, 5.4651

In [ ]:
# Next: New notebook :) https://docs.pytorch.org/tutorials/beginner/introyt/introyt_index.html

In [ ]:
optimizer.state_dict()['state'][0]
# exp_avg: needed for first moment calc
# exp_avg_sq: needed for second moment calc

{'step': tensor(1876.),
 'exp_avg': tensor([[-5.6052e-45, -7.9087e-18, -7.9087e-18,  ..., -1.7575e-05,
          -1.5960e-06, -1.1142e-08],
         [ 5.6052e-45, -4.3206e-11, -4.3206e-11,  ..., -1.9458e-05,
          -1.9863e-06, -1.3464e-08],
         [-5.6052e-45, -5.6052e-45,  6.6513e-37,  ..., -3.8257e-06,
          -4.7210e-42, -5.6052e-45],
         ...,
         [ 0.0000e+00,  5.6052e-45,  5.6052e-45,  ..., -2.8521e-11,
          -3.6365e-10, -4.5682e-43],
         [ 5.6052e-45,  1.0288e-07,  3.0089e-07,  ..., -3.9293e-07,
           1.1770e-07,  1.9176e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  5.6052e-45,
           0.0000e+00,  0.0000e+00]], device='cuda:0'),
 'exp_avg_sq': tensor([[2.8818e-14, 6.6150e-13, 6.8260e-13,  ..., 1.1444e-08, 2.3309e-09,
          1.6501e-10],
         [4.4530e-17, 7.1023e-13, 8.6959e-13,  ..., 3.7205e-08, 1.1921e-08,
          3.7147e-10],
         [1.5391e-13, 1.5463e-13, 5.4651e-12,  ..., 4.1354e-10, 5.4206e-11,
          1.74

In [ ]:
optimizer.state_dict().keys()

dict_keys(['state', 'param_groups'])